# 🎨 مكتبة الشعلة - خادم الذكاء الاصطناعي المجاني لمطابقة الوجوه 100%
## Maktaba Shula - Free InstantID / FaceID Cloud GPU Server
---
### 📖 تعليمات التشغيل السريع (3 خطوات فقط):
1. تأكد من تفعيل كرت الشاشة المجاني: من القائمة بالأعلى اختر **Runtime -> Change runtime type -> T4 GPU** ثم اضغط Save.
2. اضغط على زر **Run All** (أو اضغط `Ctrl + F9`) لتشغيل كافة الخلايا بالترتيب.
3. انتظر 2 إلى 3 دقائق، ستجد رابطاً باللون الأخضر في نهاية الخلية الأخيرة مثل:
   `👉 الرابط المخصص: https://xxxx.trycloudflare.com/generate`
   انسخ هذا الرابط والصقه في خانة **«رابط API مخصص»** في موقع مكتبة الشعلة، واستمتع بتوليد مجلات أطفال مطابقة لملامح الطفل 100%!

In [ ]:
# 1. تثبيت الحزم والمكتبات الضرورية (InstantID & FaceID Pipeline)
!pip install -q diffusers transformers accelerate insightface onnxruntime-gpu fastapi uvicorn nest_asyncio pyngrok python-multipart
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# 2. تحميل أوزان نموذج InstantID SDXL
import torch
from diffusers import StableDiffusionXLInstantIDPipeline, ControlNetModel
from insightface.app import FaceAnalysis
from PIL import Image
import io
import base64

print("⏳ جاري تحميل نموذج InstantID لكشف الوجه وتثبيت الملامح...")
app = FaceAnalysis(name='antelopev2', root='./', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

# تحميل ControlNet و InstantID Pipeline
controlnet = ControlNetModel.from_pretrained(
    "InstantX/InstantID",
    subfolder="ControlNetModel",
    torch_dtype=torch.float16
).to("cuda")

pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    "wangqixun/YamerMIX_v8",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

pipe.load_ip_adapter_instantid("InstantX/InstantID", subfolder="ip-adapter.bin")
pipe.set_ip_adapter_scale(0.85)
print("✅ تم تحميل نموذج InstantID بنجاح والـ GPU جاهز 100%!")

In [ ]:
# 3. إنشاء خادم الـ API والربط بنفق Cloudflare المجاني
import uvicorn
import nest_asyncio
import threading
import subprocess
import time
import re
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware

fastapi_app = FastAPI(title="Maktaba Shula FaceID Engine")
fastapi_app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@fastapi_app.get("/")
def root():
    return {"status": "active", "service": "Maktaba Shula InstantID Server", "gpu": torch.cuda.get_device_name(0)}

@fastapi_app.post("/generate")
async def generate(req: Request):
    data = await req.json()
    prompt = data.get("prompt", "an authentic cute child")
    negative_prompt = data.get("negative_prompt", "ugly, deformed, bad eyes")
    face_b64 = data.get("image") or data.get("child_photo_base64") or data.get("reference_image")
    
    if face_b64 and "," in face_b64:
        face_b64 = face_b64.split(",")[1]
    
    face_image = None
    if face_b64:
        face_image = Image.open(io.BytesIO(base64.b64decode(face_b64))).convert("RGB")
    
    if face_image:
        face_info = app.get(torch.tensor(face_image))
        if len(face_info) > 0:
            face_emb = face_info[0].embedding
            out = pipe(prompt=prompt, negative_prompt=negative_prompt, image_embeds=face_emb, image=face_image, num_inference_steps=25, guidance_scale=5.0).images[0]
        else:
            out = pipe(prompt=prompt, negative_prompt=negative_prompt, num_inference_steps=25).images[0]
    else:
        out = pipe(prompt=prompt, negative_prompt=negative_prompt, num_inference_steps=25).images[0]
    
    buf = io.BytesIO()
    out.save(buf, format="PNG")
    b64_str = base64.b64encode(buf.getvalue()).decode("utf-8")
    return {"success": True, "image": f"data:image/png;base64,{b64_str}"}

# تشغيل الخادم في الخلفية
nest_asyncio.apply()
def run_server():
    uvicorn.run(fastapi_app, host="127.0.0.1", port=8000, log_level="warning")
threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

# فتح نفق Cloudflare وإظهار الرابط للمستخدم
print("🌐 جاري توليد رابط الـ API السحابي المجاني...")
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tunnel_url = None
for line in p.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*70)
    print("🎉 تم تشغيل خادم InstantID المجاني بنجاح!")
    print(f"👉 رابط الـ API المخصص: {tunnel_url}/generate")
    print("انسخ الرابط أعلاه والصقه في خانة (رابط API مخصص) في موقع مكتبة الشعلة")
    print("="*70)
else:
    print("يرجى إعادة تشغيل الخلية للحصول على الرابط.")